# $\color{#dda}{\text{}}$ 

#### $\color{#dda}{\text{Segundo Examen Parcial: }}$
# $\color{#dda}{\text{Modelo predictivo de estatura}}$
### $\color{#dda}{\text{Por su servilleta: Lambda Remiel Heredia Pérez}}$ 
**Dataset:** `personas_medidas.csv` (n = 48)
**Variable objetivo:** `estatura_cm`
**Predictores:** `envergadura_cm`, `longitud_antebrazo_cm`, `longitud_mano_cm`, `talla_zapato_mx`
**Modelo elegido:** Ridge Regression (L2)

DISCLAIMER: Se excluyen `id`, `sexo` y `edad`: el proyecto define como variables predictoras
solo las cuatro medidas antropométricas. `sexo` se usa únicamente en el EDA
para justificar su efecto sobre la estatura, no como entrada del modelo.

# $\color{#dda}{\text{Imports}}$

In [18]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, Ridge, LassoCV, ElasticNetCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
pd.set_option("display.precision", 3)

In [19]:
df = pd.read_csv("personas_medidas.csv")

print("Dimensiones:", df.shape)
print("\nNulos por columna:\n", df.isna().sum())
print("\nDuplicados exactos:", df.duplicated().sum())
print("\nResumen numérico:")
display(df.describe().T)

# Chequeo de rangos plausibles
print("\nValores fuera de rango antropométrico razonable:")
print(" Estatura fuera de [140, 210]:", ((df.estatura_cm < 140) | (df.estatura_cm > 210)).sum())
print(" Envergadura fuera de [140, 215]:", ((df.envergadura_cm < 140) | (df.envergadura_cm > 215)).sum())
print(" Talla zapato MX fuera de [20, 31]:", ((df.talla_zapato_mx < 20) | (df.talla_zapato_mx > 31)).sum())

Dimensiones: (48, 8)

Nulos por columna:
 id                       0
sexo                     0
edad                     0
estatura_cm              0
envergadura_cm           0
longitud_antebrazo_cm    0
longitud_mano_cm         0
talla_zapato_mx          0
dtype: int64

Duplicados exactos: 0

Resumen numérico:


,count,mean,std,min,25%,50%,75%,max
id,48.0,24.500,14.000,1.0,12.750,24.50,36.250,48.0
edad,48.0,25.708,5.874,18.0,20.750,25.50,30.000,37.0
estatura_cm,48.0,165.746,8.890,145.8,159.125,165.35,171.900,186.8
envergadura_cm,48.0,166.525,9.784,145.8,160.850,166.70,173.100,185.2
longitud_antebrazo_cm,48.0,25.669,1.830,22.6,24.375,25.45,26.775,29.5
longitud_mano_cm,48.0,17.365,1.131,15.1,16.675,17.30,18.125,19.9
talla_zapato_mx,48.0,24.448,2.102,21.0,22.875,24.25,26.125,28.5



Valores fuera de rango antropométrico razonable:
 Estatura fuera de [140, 210]: 0
 Envergadura fuera de [140, 215]: 0
 Talla zapato MX fuera de [20, 31]: 0


## $\color{#dda}{\text{Limpieza}}$

- No hay valores nulos ni duplicados exactos -> no se imputa ni se eliminan filas.
- Todas las variables numéricas caen en rangos antropométricos plausibles;
  no hay errores de captura evidentes (p. ej. estatura en metros en vez de cm).
- **No se eliminan outliers**: la variabilidad real de estaturas (145.8–186.8 cm)
  es información legítima, y con n=48 eliminar casos sesga el modelo.
  Además, Ridge con estandarización es razonablemente robusto a valores extremos
  moderados.
- `sexo` y `edad` se conservan en el dataframe para el EDA, pero **no entran al modelo**.

In [20]:
fig = px.box(
    df, x="sexo", y="estatura_cm", color="sexo",
    points="all", hover_data=["edad", "envergadura_cm"],
    title="Estatura por sexo (EDA exploratorio, no es predictor del modelo)"
)
fig.update_layout(showlegend=False)
fig.show()

### $\color{#dda}{\text{¿Por qué el sexo afecta la estatura?}}$

El dimorfismo sexual humano produce medias de estatura distintas (~13 cm de diferencia típica en población mexicana adulta). Se observa que el grupo F se concentra por debajo de ~170 cm y el grupo M se extiende hasta ~187 cm. Esto confirma que `sexo` es informativo, **pero el proyecto restringe los predictores a las cuatro medidas**, así que se documenta como limitación (el modelo debe funcionar midiendo solo segmentos corporales, sin preguntar sexo).

In [21]:
vars_modelo = ["estatura_cm", "envergadura_cm", "longitud_antebrazo_cm",
               "longitud_mano_cm", "talla_zapato_mx"]

fig = px.scatter_matrix(
    df, dimensions=vars_modelo, color="sexo",
    title="Matriz de dispersión",
    height=800, opacity=0.75
)
fig.update_traces(diagonal_visible=False, marker=dict(size=6))
fig.show()

In [22]:
corr = df[vars_modelo].corr(numeric_only=True)

fig = px.imshow(
    corr, text_auto=".3f", color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1, title="Correlación de Pearson"
)
fig.update_layout(width=700, height=600)
fig.show()

print(corr["estatura_cm"].sort_values(ascending=False))

estatura_cm              1.000
envergadura_cm           0.942
talla_zapato_mx          0.933
longitud_antebrazo_cm    0.785
longitud_mano_cm         0.747
Name: estatura_cm, dtype: float64


### $\color{#dda}{\text{Hallazgos del EDA}}$

1. `envergadura_cm` es el predictor dominante (r ≈ 0.95 con la estatura):
   la envergadura y la estatura comparten el mismo esqueleto axial/apendicular.
2. `longitud_antebrazo_cm` y `talla_zapato_mx` correlacionan moderadamente
   (r ≈ 0.6–0.8) y **entre sí** (r ≈ 0.7–0.8) → **multicolinealidad**.
3. `longitud_mano_cm` es el predictor más débil (r ≈ 0.5–0.6) y aporta poco
   después de controlar por envergadura.
4. La relación es aproximadamente **lineal** (nubes alargadas sin curvatura),
   lo que justifica un modelo lineal regularizado y no un kernel.
5. La multicolinealidad del punto 2 es exactamente el escenario donde
   Ridge brilla: reparte el peso entre variables correlacionadas en vez de
   producir coeficientes inestables de signo arbitrario (lo que haría OLS).

In [23]:
X = df[["envergadura_cm", "longitud_antebrazo_cm", "longitud_mano_cm", "talla_zapato_mx"]]
y = df["estatura_cm"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape[0]} casos | Test: {X_test.shape[0]} casos")
print(f"Media estatura train: {y_train.mean():.2f} cm | test: {y_test.mean():.2f} cm")
print(f"Desv. est. train: {y_train.std():.2f} cm")

Train: 38 casos | Test: 10 casos
Media estatura train: 166.05 cm | test: 164.60 cm
Desv. est. train: 9.28 cm


### $\color{#dda}{\text{Función objetivo y regularización (Ridge)}}$

Ridge minimiza el error cuadrático con una penalización L2 sobre los coeficientes:

$$
J(\mathbf{w}) \;=\;
\underbrace{\sum_{i=1}^{n} \bigl(y_i - \hat{y}_i\bigr)^{2}}_{\text{error de ajuste}}
\;+\;
\alpha \,
\underbrace{\sum_{j=1}^{p} w_j^{2}}_{\text{regularización}}
$$

El intercepto w₀ no se penalizacon $\hat{y}_i \;=\; w_0 \;+\; \sum_{j=1}^{p} w_j \, x_{ij}$.

**Por qué L2 y no L1 (Lasso):**
- Los cuatro predictores son medidas de tamaño corporal y están correlacionados.
  Con L1, el modelo tiende a conservar uno y anular los demás de forma arbitraria
  (inestable con n pequeño). Con L2, el peso se reparte suavemente entre
  predictores correlacionados -> solución estable.
- L2 no lleva coeficientes exactamente a cero, pero **encoge** su magnitud,
  reduciendo la varianza del estimador a cambio de un poco de sesgo.
  Con 38 casos de entrenamiento y 4 predictores ese intercambio conviene.

**Estandarización obligatoria:** la penalización NO es invariante a escala.
`talla_zapato_mx` (~26) y `longitud_mano_cm` (~17) están en escalas distintas;
sin `StandardScaler` el castigo recaería desproporcionadamente sobre las
variables de mayor magnitud. Por eso el modelo es un `Pipeline(StandardScaler → Ridge)`.

**Selección de α:** `RidgeCV` con validación cruzada de 5 folds sobre el
conjunto de entrenamiento, probando α en una grilla logarítmica de 10⁻³ a 10³.

In [24]:
modelo = Pipeline([
    ("escalador", StandardScaler()),
    ("ridge", RidgeCV(
        alphas=np.logspace(-3, 3, 200),
        cv=KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    ))
])

modelo.fit(X_train, y_train)

alpha_opt = modelo.named_steps["ridge"].alpha_
print(f"α óptimo encontrado por CV: {alpha_opt:.4f}")
print("Coeficientes (en escala estandarizada):")
for nombre, coef in zip(X.columns, modelo.named_steps["ridge"].coef_):
    print(f"  {nombre:24s}: {coef:7.3f}")
print(f"  {'intercepto':24s}: {modelo.named_steps['ridge'].intercept_:7.3f}")

α óptimo encontrado por CV: 0.6368
Coeficientes (en escala estandarizada):
  envergadura_cm          :   4.328
  longitud_antebrazo_cm   :   0.844
  longitud_mano_cm        :   0.938
  talla_zapato_mx         :   3.314
  intercepto              : 166.047


In [25]:
pred_train = modelo.predict(X_train)
pred_test  = modelo.predict(X_test)

# --- Modelo ---
mae_train = mean_absolute_error(y_train, pred_train)
mae_test  = mean_absolute_error(y_test,  pred_test)
rmse_test = np.sqrt(mean_squared_error(y_test, pred_test))
r2_test   = r2_score(y_test, pred_test)

# --- Baseline: predecir siempre la media del entrenamiento ---
media_train = y_train.mean()
pred_baseline = np.full(len(y_test), media_train)
mae_baseline  = mean_absolute_error(y_test, pred_baseline)

print("="*58)
print(f"{'Métrica':<28}{'Modelo Ridge':>14}{'Baseline':>14}")
print("-"*58)
print(f"{'MAE train (cm)':<28}{mae_train:>14.3f}{'—':>14}")
print(f"{'MAE test (cm)':<28}{mae_test:>14.3f}{mae_baseline:>14.3f}")
print(f"{'RMSE test (cm)':<28}{rmse_test:>14.3f}{'-':>14}")
print(f"{'R² test':<28}{r2_test:>14.3f}{'-':>14}")
print("="*58)
print(f"Media del train usada por el baseline: {media_train:.2f} cm")
print(f"Reducción del error vs baseline: {100*(1 - mae_test/mae_baseline):.1f}%")

Métrica                       Modelo Ridge      Baseline
----------------------------------------------------------
MAE train (cm)                       1.738             —
MAE test (cm)                        1.392         6.040
RMSE test (cm)                       1.658             -
R² test                              0.946             -
Media del train usada por el baseline: 166.05 cm
Reducción del error vs baseline: 77.0%


In [26]:
lims = [min(y_test.min(), pred_test.min()) - 3,
        max(y_test.max(), pred_test.max()) + 3]

fig = go.Figure()

# Banda de +-1 MAE
fig.add_trace(go.Scatter(
    x=[lims[0], lims[1], lims[1], lims[0]],
    y=[lims[0]-mae_test, lims[1]-mae_test, lims[1]+mae_test, lims[0]+mae_test],
    fill="toself", fillcolor="rgba(99,110,250,0.10)",
    line=dict(color="rgba(0,0,0,0)"), name=f"Banda ±MAE ({mae_test:.2f} cm)"
))

# Línea y = x (predicción perfecta)
fig.add_trace(go.Scatter(x=lims, y=lims, mode="lines", name="y = x",
                         line=dict(dash="dash", color="gray", width=2)))

# Puntos del modelo
fig.add_trace(go.Scatter(
    x=y_test, y=pred_test, mode="markers",
    marker=dict(size=11, color="#636EFA", line=dict(width=1, color="white")),
    name="Ridge", text=[f"error: {p-r:+.1f} cm" for r, p in zip(y_test, pred_test)],
    hovertemplate="Real: %{x:.1f} cm<br>Predicho: %{y:.1f} cm<br>%{text}<extra></extra>"
))

# Puntos del baseline
fig.add_trace(go.Scatter(
    x=y_test, y=pred_baseline, mode="markers",
    marker=dict(size=9, symbol="x", color="#EF553B"),
    name=f"Baseline (media = {media_train:.1f} cm)"
))

fig.add_annotation(x=0.03, y=0.97, xref="paper", yref="paper", showarrow=False,
                   text=f"<b>MAE Ridge = {mae_test:.2f} cm</b><br>MAE Baseline = {mae_baseline:.2f} cm",
                   align="left", bgcolor="white", bordercolor="gray", borderwidth=1)

fig.update_layout(
    title="Valores reales vs predichos: conjunto de prueba (n=10)",
    xaxis_title="Estatura real (cm)",
    yaxis_title="Estatura predicha (cm)",
    xaxis=dict(range=lims), yaxis=dict(range=lims),
    width=750, height=650, template="plotly_white"
)
fig.show()

In [27]:
from sklearn.linear_model import Lasso, ElasticNet
from sklearn.kernel_ridge import KernelRidge

modelos = {
    "Ridge (L2)":      Pipeline([("s", StandardScaler()), ("m", Ridge(alpha=alpha_opt))]),
    "Lasso (L1)":      Pipeline([("s", StandardScaler()), ("m", Lasso(alpha=0.5, max_iter=50000))]),
    "Elastic Net":     Pipeline([("s", StandardScaler()), ("m", ElasticNet(alpha=0.5, l1_ratio=0.5, max_iter=50000))]),
    "KRR (RBF)":       Pipeline([("s", StandardScaler()), ("m", KernelRidge(kernel="rbf", alpha=1.0, gamma=0.1))]),
}

filas = []
for nombre, m in modelos.items():
    m.fit(X_train, y_train)
    filas.append({
        "Modelo": nombre,
        "MAE train": mean_absolute_error(y_train, m.predict(X_train)),
        "MAE test":  mean_absolute_error(y_test,  m.predict(X_test)),
    })

tabla = pd.DataFrame(filas).sort_values("MAE test")
display(tabla.style.format({"MAE train": "{:.3f}", "MAE test": "{:.3f}"})
        .highlight_min(subset=["MAE test"], color="lightgreen"))

print("Baseline (media del train) MAE test:", round(mae_baseline, 3))

,Modelo,MAE train,MAE test
1,Lasso (L1),1.774,1.275
0,Ridge (L2),1.738,1.392
2,Elastic Net,1.814,1.473
3,KRR (RBF),10.415,14.512


Baseline (media del train) MAE test: 6.04


### $\color{#dda}{\text{Export}}$

In [34]:
import joblib
from pathlib import Path
from datetime import datetime
import sklearn

RUTA_MODELO = Path("modelo_ridge_estatura.joblib")
RUTA_META   = Path("modelo_ridge_estatura_meta.joblib")

joblib.dump(modelo, RUTA_MODELO, compress=3)

meta = {
    "fecha_entrenamiento": datetime.now().isoformat(timespec="seconds"),
    "columnas_X": list(X.columns),
    "alpha_optimo": float(modelo.named_steps["ridge"].alpha_),
    "media_train": float(y_train.mean()),
    "mae_train": float(mean_absolute_error(y_train, modelo.predict(X_train))),
    "mae_test":  float(mae_test),
    "mae_baseline": float(mae_baseline),
    "n_train": int(X_train.shape[0]),
    "n_test":  int(X_test.shape[0]),
    "sklearn_version": sklearn.__version__,
}
joblib.dump(meta, RUTA_META, compress=3)

print(f"✔ Modelo exportado: {RUTA_MODELO.resolve()}")
print(f"✔ Metadatos       : {RUTA_META.resolve()}")
print(f"  α óptimo        : {meta['alpha_optimo']:.4f}")
print(f"  MAE test        : {meta['mae_test']:.3f} cm")
print(f"  MAE baseline    : {meta['mae_baseline']:.3f} cm")

✔ Modelo exportado: C:\Users\hecto\OneDrive\Escritorio\ITESO\6to Semestre\EO\Exámenes\modelo_ridge_estatura.joblib
✔ Metadatos       : C:\Users\hecto\OneDrive\Escritorio\ITESO\6to Semestre\EO\Exámenes\modelo_ridge_estatura_meta.joblib
  α óptimo        : 0.6368
  MAE test        : 1.392 cm
  MAE baseline    : 6.040 cm
